In [ ]:
!pip install nvidia-dali-cuda120

In [ ]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass

test = TestClass()
test.test()          # IT'S WORKING!

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 123 (delta 56), reused 86 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 2.97 MiB | 11.51 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Github classes initialized!


In [ ]:
!git -C /content/yolov1-cupy checkout darknet

Branch 'darknet' set up to track remote branch 'darknet' from 'origin'.
Switched to a new branch 'darknet'


Downloading the ImageNet10 dataset from kaggle

In [ ]:
!curl -L -o /content/imagenet10.zip https://www.kaggle.com/api/v1/datasets/download/liusha249/imagenet10
print("Extracting dataset (quiet unzip)...")
!unzip -q /content/imagenet10.zip -d /content/yolov1-cupy
print("Done.")
!rm /content/imagenet10.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1304M  100 1304M    0     0   247M      0  0:00:05  0:00:05 --:--:--  257M
Extracting dataset (quiet unzip)...
Done.


## Pretraining (Darknet, ImageNet10)

A few epochs of SGD over the full dataset. Each epoch uses a new shuffle (`seed = epoch`). Re-run after dataset + path cells.

In [ ]:
# Fast data-preprocessor, all done on GPU

from pathlib import Path
import math
import random
import numpy as np
import cupy as cp

from nvidia.dali import fn, types, pipeline_def

_IMAGE_EXT = frozenset({".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG"})


def _scan_imagefolder(root):
    root = Path(root)
    paths = sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix in _IMAGE_EXT
    )
    if not paths:
        raise FileNotFoundError(f"No images found under {root}")

    class_names = sorted({p.parent.name for p in paths})
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    labeled = [(str(p), class_to_idx[p.parent.name]) for p in paths]
    return labeled, class_to_idx


def random_split_labeled(labeled, lengths, seed=42):
    """
    lengths:
      - tuple/list of ints summing to len(labeled), or
      - tuple/list of floats summing to 1.0
    """
    n = len(labeled)

    if all(isinstance(x, float) for x in lengths):
        assert abs(sum(lengths) - 1.0) < 1e-8, f"Float split must sum to 1, got {sum(lengths)}"
        counts = [int(n * x) for x in lengths]
        counts[-1] = n - sum(counts[:-1])
    else:
        counts = list(lengths)
        assert sum(counts) == n, f"Split sizes must sum to {n}, got {sum(counts)}"

    idx = list(range(n))
    rng = random.Random(seed)
    rng.shuffle(idx)

    parts = []
    start = 0
    for c in counts:
        sub_idx = idx[start:start + c]
        parts.append([labeled[i] for i in sub_idx])
        start += c
    return parts


def write_file_list(items, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for path, label in items:
            f.write(f"{path} {label}\n")
    return str(out_path)


def prepare_split_file_lists(
    data_root,
    split,
    *,
    seed=42,
    cache_dir=None,
):
    """
    split can be:
      - (train_count, val_count)
      - (train_fraction, val_fraction)
    """
    labeled, class_to_idx = _scan_imagefolder(data_root)
    train_items, val_items = random_split_labeled(labeled, split, seed=seed)

    if cache_dir is None:
        cache_dir = Path(data_root) / ".dali_splits"
    else:
        cache_dir = Path(cache_dir)

    train_list = write_file_list(train_items, cache_dir / f"train_seed{seed}.lst")
    val_list = write_file_list(val_items, cache_dir / f"val_seed{seed}.lst")

    return {
        "train_file_list": train_list,
        "val_file_list": val_list,
        "num_train": len(train_items),
        "num_val": len(val_items),
        "class_to_idx": class_to_idx,
    }


def num_batches(n_samples, batch_size, drop_last=False):
    return n_samples // batch_size if drop_last else math.ceil(n_samples / batch_size)


@pipeline_def
def imagefolder_pipe(
    file_list: str,
    image_size: int = 224,
    training: bool = True,
):
    images, labels = fn.readers.file(
        file_list=file_list,
        random_shuffle=False,
        shuffle_after_epoch=training,
        name="Reader",
    )

    images = fn.decoders.image(
        images,
        device="mixed",
        output_type=types.RGB,
    )

    images = fn.resize(
        images,
        device="gpu",
        resize_x=image_size,
        resize_y=image_size,
        interp_type=types.INTERP_LINEAR,
    )

    images = fn.crop_mirror_normalize(
        images,
        device="gpu",
        dtype=types.FLOAT,
        output_layout="CHW",
        crop=(image_size, image_size),
        mean=[0.0, 0.0, 0.0],
        std=[255.0, 255.0, 255.0],
    )

    return images, labels


def build_train_val_pipelines(
    data_root,
    *,
    split,
    batch_size=8,
    image_size=224,
    split_seed=42,
    num_threads=4,
    device_id=0,
    dali_seed=42,
    cache_dir=None,
):
    meta = prepare_split_file_lists(
        data_root,
        split,
        seed=split_seed,
        cache_dir=cache_dir,
    )

    train_pipe = imagefolder_pipe(
        batch_size=batch_size,
        num_threads=num_threads,
        device_id=device_id,
        seed=dali_seed,
        file_list=meta["train_file_list"],
        image_size=image_size,
        training=True,
    )
    train_pipe.build()

    val_pipe = imagefolder_pipe(
        batch_size=batch_size,
        num_threads=num_threads,
        device_id=device_id,
        seed=dali_seed,
        file_list=meta["val_file_list"],
        image_size=image_size,
        training=False,
    )
    val_pipe.build()

    return train_pipe, val_pipe, meta


def _to_cupy(x):
    try:
        return cp.asarray(x)
    except Exception:
        pass

    try:
        return cp.asarray(x.as_cpu().as_array())
    except Exception:
        pass

    raise TypeError(f"Could not convert DALI output to CuPy: {type(x)}")


def dali_batch_to_xy(pipe):
    images, labels = pipe.run()
    x = _to_cupy(images)
    y = labels.as_array().reshape(-1).astype(np.int64, copy=False)
    return x, y

In [ ]:
from pathlib import Path
import time

import cupy as cp
from tqdm.auto import tqdm

from cross_entropy import softmax_cross_entropy_grad, softmax_cross_entropy_loss
from darknet import Darknet


REPO = "/content/yolov1-cupy"
DATA_ROOT_PATH = f"{REPO}/imagenet-10"
BATCH_SIZE = 8
LEARNING_RATE = 0.01
NUM_EPOCHS = 20
TRAIN_TEST_SPLIT = (0.8, 0.2)
DROP_LAST = True

model = Darknet(num_classes=10)

train_pipe, val_pipe, meta = build_train_val_pipelines(
    DATA_ROOT_PATH,
    split=TRAIN_TEST_SPLIT,
    batch_size=BATCH_SIZE,
    image_size=224,
    split_seed=42,
    num_threads=4,
    device_id=0,
    dali_seed=42,
)

n_train = meta["num_train"]
n_val = meta["num_val"]
n_train_batches = num_batches(n_train, BATCH_SIZE, drop_last=DROP_LAST)
n_val_batches = num_batches(n_val, BATCH_SIZE, drop_last=DROP_LAST)

print(f"train samples: {n_train}, train batches: {n_train_batches}")
print(f"val samples:   {n_val}, val batches:   {n_val_batches}")
print(meta["class_to_idx"])

for epoch in range(NUM_EPOCHS):
    # ---- train ----
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0

    pbar = tqdm(
        range(n_train_batches),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [train]",
        unit="batch",
        mininterval=0.5,
    )

    for batch_idx in pbar:
        dp_start_time = time.perf_counter_ns()
        x, y_cpu = dali_batch_to_xy(train_pipe)
        dp_end_time = time.perf_counter_ns()

        # mimic drop_last=True
        if x.shape[0] != BATCH_SIZE:
            break

        model.zero_grad()
        logits = model.forward(x)

        y_gpu = cp.asarray(y_cpu, dtype=cp.int64)
        pred = cp.argmax(logits, axis=1)
        epoch_correct += int(cp.sum(pred == y_gpu))
        epoch_total += int(y_gpu.shape[0])

        batch_loss = softmax_cross_entropy_loss(
            logits, y_cpu, mean_over_batch=True
        )
        epoch_loss += batch_loss

        grad_logits = softmax_cross_entropy_grad(
            logits, y_cpu, mean_over_batch=True
        )
        model.backward(grad_logits)
        model.sgd_step(LEARNING_RATE)

        running_mean = epoch_loss / (batch_idx + 1)
        running_acc = epoch_correct / epoch_total

        fb_end_time = time.perf_counter_ns()
        pbar.set_postfix_str(
            f"loss={running_mean:.4f} acc={running_acc:.3f} "
            f"dp={(dp_end_time - dp_start_time)/1e9:.3f}s "
            f"fb={(fb_end_time - dp_end_time)/1e9:.3f}s"
        )

    train_pipe.reset()

    train_loss = epoch_loss / max(1, n_train_batches)
    train_acc = epoch_correct / max(1, epoch_total)

    # ---- validation ----
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    vbar = tqdm(
        range(n_val_batches),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [val]",
        unit="batch",
        mininterval=0.5,
    )

    for batch_idx in vbar:
        x, y_cpu = dali_batch_to_xy(val_pipe)

        if x.shape[0] != BATCH_SIZE:
            break

        logits = model.forward(x)

        y_gpu = cp.asarray(y_cpu, dtype=cp.int64)
        pred = cp.argmax(logits, axis=1)
        val_correct += int(cp.sum(pred == y_gpu))
        val_total += int(y_gpu.shape[0])

        batch_loss = softmax_cross_entropy_loss(
            logits, y_cpu, mean_over_batch=True
        )
        val_loss += batch_loss

        running_val_loss = val_loss / (batch_idx + 1)
        running_val_acc = val_correct / val_total
        vbar.set_postfix_str(
            f"loss={running_val_loss:.4f} acc={running_val_acc:.3f}"
        )

    val_pipe.reset()

    mean_val_loss = val_loss / max(1, n_val_batches)
    val_acc = val_correct / max(1, val_total)

    print(
        f"epoch {epoch + 1}/{NUM_EPOCHS} done — "
        f"train loss {train_loss:.4f} train acc {train_acc:.4f} | "
        f"val loss {mean_val_loss:.4f} val acc {val_acc:.4f}"
    )

weights_path = Path(REPO) / "darknet_pretrained.npz"
model.save_weights(weights_path)
print(f"saved weights to {weights_path.resolve()}")

train samples: 10400, train batches: 1300
val samples:   2600, val batches:   325
{'n02056570': 0, 'n02085936': 1, 'n02128757': 2, 'n02690373': 3, 'n02692877': 4, 'n03095699': 5, 'n04254680': 6, 'n04285008': 7, 'n04467665': 8, 'n07747607': 9}


Epoch 1/20 [train]:   0%|          | 0/1300 [00:00<?, ?batch/s]

Epoch 1/20 [val]:   0%|          | 0/325 [00:00<?, ?batch/s]

epoch 1/20 done — train loss 1.5560 train acc 0.4441 | val loss 1.3211 val acc 0.5408


Epoch 2/20 [train]:   0%|          | 0/1300 [00:00<?, ?batch/s]

Epoch 2/20 [val]:   0%|          | 0/325 [00:00<?, ?batch/s]

epoch 2/20 done — train loss 1.0899 train acc 0.6309 | val loss 1.0984 val acc 0.6354


Epoch 3/20 [train]:   0%|          | 0/1300 [00:00<?, ?batch/s]

KeyboardInterrupt: 